# Project 2: Simplified Flash Attention

**What we build:** A CUDA kernel that computes exact attention without materialising the N×N attention matrix in HBM (slow GPU memory). Based on the FlashAttention paper (Dao et al., 2022).

**Three kernels:**
1. `naive_attention` — computes full `[N×N]` attention matrix in global memory
2. `online_softmax` — one-pass numerically stable softmax with warp-shuffle reduction
3. `flash_attention_fwd` — tiled attention: only `O(Nd)` memory, `Br×Bc` tiles live in SRAM

**Key insight:** For N=512, d=64, 8 heads → the attention matrix alone is **1MB per layer**. For GPT-3 (96 layers, 96 heads): **8.8 GB just for attention scores**. Flash attention avoids all of it.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nvcc --version | grep release
import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Shared mem/SM: {torch.cuda.get_device_properties(0).shared_memory_per_block / 1024:.0f} KB')

In [ ]:
# ── Cell 2: Write all source files ───────────────────────────────────
import os
os.makedirs('flash_attn', exist_ok=True)
os.chdir('flash_attn')

# ── flash_attn_kernels.cu ──────────────────────────────────────────
kernels_code = r'''
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <float.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define Br        16
#define Bc        16
#define HEAD_DIM  64
#define WARP_SIZE 32

#define CUDA_CHECK(x) do { cudaError_t e=(x); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e));exit(1);} } while(0)
#define CUBLAS_CHECK(x) do { cublasStatus_t s=(x); \
    if(s!=CUBLAS_STATUS_SUCCESS){fprintf(stderr,"cuBLAS %s:%d %d\n",__FILE__,__LINE__,(int)s);exit(1);} } while(0)

/* ── KERNEL 1: Naive attention ── compute S = QK^T / sqrt(d) ────── */
__global__ void naive_attention(
    const float* __restrict__ Q, const float* __restrict__ K,
    const float* __restrict__ V, float* __restrict__ O,
    float* __restrict__ S, int N, int d)
{
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int j = blockIdx.y*blockDim.y + threadIdx.y;
    if (i>=N || j>=N) return;
    float scale = rsqrtf((float)d), s=0.f;
    for (int k=0;k<d;++k) s += Q[i*d+k]*K[j*d+k];
    S[i*N+j] = s*scale;
}

__global__ void naive_softmax_v(
    const float* __restrict__ S, const float* __restrict__ V,
    float* __restrict__ O, int N, int d)
{
    int i = blockIdx.x;
    if (i>=N) return;
    float row_max=-FLT_MAX;
    for (int j=0;j<N;++j) row_max=fmaxf(row_max,S[i*N+j]);
    float denom=0.f;
    for (int j=0;j<N;++j) denom+=expf(S[i*N+j]-row_max);
    for (int k=threadIdx.x;k<d;k+=blockDim.x) {
        float out=0.f;
        for (int j=0;j<N;++j) out+=expf(S[i*N+j]-row_max)/denom*V[j*d+k];
        O[i*d+k]=out;
    }
}

/* ── KERNEL 2: Online softmax (standalone demo) ──────────────────── */
__global__ void online_softmax(
    const float* __restrict__ in, float* __restrict__ out, int B, int N)
{
    int batch=blockIdx.x; if(batch>=B) return;
    const float* row_in=in+batch*N; float* row_out=out+batch*N;
    float m=-FLT_MAX, l=0.f;
    for (int j=threadIdx.x;j<N;j+=blockDim.x) {
        float x=row_in[j], mn=fmaxf(m,x);
        l=l*expf(m-mn)+expf(x-mn); m=mn;
    }
    for (int offset=WARP_SIZE/2;offset>0;offset>>=1) {
        float m2=__shfl_down_sync(0xFFFFFFFF,m,offset);
        float l2=__shfl_down_sync(0xFFFFFFFF,l,offset);
        float mn=fmaxf(m,m2); l=l*expf(m-mn)+l2*expf(m2-mn); m=mn;
    }
    __shared__ float sm[1],sl[1];
    if(threadIdx.x==0){sm[0]=m;sl[0]=l;}
    __syncthreads();
    float gm=sm[0],gl=sl[0];
    for (int j=threadIdx.x;j<N;j+=blockDim.x)
        row_out[j]=expf(row_in[j]-gm)/gl;
}

/* ── KERNEL 3: Flash Attention Forward ──────────────────────────── */
/*  One block per (batch*head, Q-tile).                              */
/*  SRAM layout: sQ[Br,d] | sK[Bc,d] | sV[Bc,d] | sS[Br,Bc]        */
__global__ void flash_attention_fwd(
    const float* __restrict__ Q, const float* __restrict__ K,
    const float* __restrict__ V, float* __restrict__ O,
    float* __restrict__ L, int N, int d, int num_heads)
{
    int bh=blockIdx.y, q_tile=blockIdx.x;
    int tx=threadIdx.x;
    int rit=tx/Bc, cit=tx%Bc;          /* row/col in tile */
    int q_row=q_tile*Br+rit;
    int bh_off=bh*N*d;

    extern __shared__ float smem[];
    float* sQ=smem, *sK=sQ+Br*d, *sV=sK+Bc*d, *sS=sV+Bc*d;

    /* load Q tile (reused for all K/V tiles) */
    for (int e=tx;e<Br*d;e+=blockDim.x) {
        int r=e/d,c=e%d; int gr=q_tile*Br+r;
        sQ[r*d+c]=(gr<N)?Q[bh_off+gr*d+c]:0.f;
    }
    __syncthreads();

    float m_i=-FLT_MAX, l_i=0.f;
    float O_i[HEAD_DIM]={0};

    int num_kv=(N+Bc-1)/Bc;
    for (int kv=0;kv<num_kv;++kv) {
        /* load K tile */
        for (int e=tx;e<Bc*d;e+=blockDim.x) {
            int r=e/d,c=e%d; int gr=kv*Bc+r;
            sK[r*d+c]=(gr<N)?K[bh_off+gr*d+c]:0.f;
        }
        /* load V tile */
        for (int e=tx;e<Bc*d;e+=blockDim.x) {
            int r=e/d,c=e%d; int gr=kv*Bc+r;
            sV[r*d+c]=(gr<N)?V[bh_off+gr*d+c]:0.f;
        }
        __syncthreads();

        /* compute scores for this tile */
        if (rit<Br && cit<Bc) {
            float scale=rsqrtf((float)d), s=0.f;
            #pragma unroll
            for (int k=0;k<HEAD_DIM;++k) s+=sQ[rit*d+k]*sK[cit*d+k];
            sS[rit*Bc+cit]=s*scale;
        }
        __syncthreads();

        /* update running (m, l, O) — only col 0 thread per row */
        if (cit==0 && q_row<N) {
            float m_t=-FLT_MAX;
            for (int j=0;j<Bc;++j) { int c=kv*Bc+j; if(c<N) m_t=fmaxf(m_t,sS[rit*Bc+j]); }
            float m_new=fmaxf(m_i,m_t);
            float sc_old=expf(m_i-m_new);
            l_i*=sc_old;
            #pragma unroll
            for (int c=0;c<HEAD_DIM;++c) O_i[c]*=sc_old;
            float l_t=0.f;
            for (int j=0;j<Bc;++j) {
                int kvc=kv*Bc+j; if(kvc>=N) continue;
                float p=expf(sS[rit*Bc+j]-m_new); l_t+=p;
                #pragma unroll
                for (int c=0;c<HEAD_DIM;++c) O_i[c]+=p*sV[j*d+c];
            }
            l_i+=l_t; m_i=m_new;
        }
        __syncthreads();
    }

    if (cit==0 && q_row<N) {
        float inv_l=1.f/l_i;
        #pragma unroll
        for (int c=0;c<HEAD_DIM;++c) O[bh_off+q_row*d+c]=O_i[c]*inv_l;
        if (L) L[bh*N+q_row]=logf(l_i)+m_i;
    }
}

/* ── timing helpers ──────────────────────────────────────────────── */
static float gpu_ms(cudaEvent_t s,cudaEvent_t e){
    float ms; cudaEventSynchronize(e); cudaEventElapsedTime(&ms,s,e); return ms;
}
static float max_err(const float*a,const float*b,int n){
    float me=0.f; for(int i=0;i<n;++i){float e=fabsf(a[i]-b[i]);if(e>me)me=e;} return me;
}

static void benchmark(int N,int d,int B,int H,
    cudaEvent_t evs,cudaEvent_t eve,int warmup,int reps,FILE*csv)
{
    size_t qkv=(size_t)B*H*N*d*sizeof(float);
    size_t sb=(size_t)B*H*N*N*sizeof(float);
    float *hQ=(float*)malloc(qkv),*hK=(float*)malloc(qkv),*hV=(float*)malloc(qkv);
    float *hOn=(float*)malloc(qkv),*hOf=(float*)malloc(qkv);
    srand(42);
    for(int i=0;i<B*H*N*d;++i){hQ[i]=((float)rand()/RAND_MAX-.5f)*.1f;hK[i]=hQ[i];hV[i]=hQ[i];}
    float *dQ,*dK,*dV,*dO,*dS,*dL;
    CUDA_CHECK(cudaMalloc(&dQ,qkv));CUDA_CHECK(cudaMalloc(&dK,qkv));
    CUDA_CHECK(cudaMalloc(&dV,qkv));CUDA_CHECK(cudaMalloc(&dO,qkv));
    CUDA_CHECK(cudaMalloc(&dS,sb)); CUDA_CHECK(cudaMalloc(&dL,(size_t)B*H*N*sizeof(float)));
    CUDA_CHECK(cudaMemcpy(dQ,hQ,qkv,cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dK,hK,qkv,cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dV,hV,qkv,cudaMemcpyHostToDevice));
    int tbh=B*H;
    dim3 bqk(16,16),gqk((N+15)/16,(N+15)/16);
    float ms;

    /* naive */
    for(int i=0;i<warmup;++i) for(int bh=0;bh<tbh;++bh){
        naive_attention<<<gqk,bqk>>>(dQ+bh*N*d,dK+bh*N*d,dV+bh*N*d,dO+bh*N*d,dS+bh*N*N,N,d);
        naive_softmax_v<<<N,64>>>(dS+bh*N*N,dV+bh*N*d,dO+bh*N*d,N,d);
    }
    cudaDeviceSynchronize();
    cudaEventRecord(evs);
    for(int r=0;r<reps;++r) for(int bh=0;bh<tbh;++bh){
        naive_attention<<<gqk,bqk>>>(dQ+bh*N*d,dK+bh*N*d,dV+bh*N*d,dO+bh*N*d,dS+bh*N*N,N,d);
        naive_softmax_v<<<N,64>>>(dS+bh*N*N,dV+bh*N*d,dO+bh*N*d,N,d);
    }
    cudaEventRecord(eve); ms=gpu_ms(evs,eve)/reps;
    cudaMemcpy(hOn,dO,qkv,cudaMemcpyDeviceToHost);
    float naive_ms=ms;
    printf("  Naive  N=%4d: %8.3f ms   mem=%.0f MB\n",N,ms,(double)sb/1e6);

    /* flash */
    int qt=(N+Br-1)/Br;
    dim3 fg(qt,tbh),fb(Br*Bc);
    size_t smem=(size_t)(Br*d+Bc*d+Bc*d+Br*Bc)*sizeof(float);
    for(int i=0;i<warmup;++i) flash_attention_fwd<<<fg,fb,smem>>>(dQ,dK,dV,dO,dL,N,d,H);
    cudaDeviceSynchronize();
    cudaEventRecord(evs);
    for(int r=0;r<reps;++r) flash_attention_fwd<<<fg,fb,smem>>>(dQ,dK,dV,dO,dL,N,d,H);
    cudaEventRecord(eve); ms=gpu_ms(evs,eve)/reps;
    cudaMemcpy(hOf,dO,qkv,cudaMemcpyDeviceToHost);
    float err=max_err(hOn,hOf,B*H*N*d);
    printf("  Flash  N=%4d: %8.3f ms   mem=O(Nd)  err=%.2e  speedup=%.2fx%s\n",
           N,ms,err,naive_ms/ms,ms<naive_ms?"  << FASTER":"");
    fprintf(csv,"%d,%d,%.4f,%.4f,%.3f\n",N,d,naive_ms,ms,naive_ms/ms);

    cudaFree(dQ);cudaFree(dK);cudaFree(dV);cudaFree(dO);cudaFree(dS);cudaFree(dL);
    free(hQ);free(hK);free(hV);free(hOn);free(hOf);
}

int main(void){
    cudaDeviceProp p; cudaGetDeviceProperties(&p,0);
    printf("GPU: %s  SMs: %d  SharedMem: %.0f KB\n",p.name,p.multiProcessorCount,p.sharedMemPerBlock/1024.f);
    printf("Tile: Br=%d Bc=%d d=%d  SRAM: %.1f KB\n\n",Br,Bc,HEAD_DIM,
           (Br*HEAD_DIM+Bc*HEAD_DIM+Bc*HEAD_DIM+Br*Bc)*4.f/1024.f);
    FILE* csv=fopen("flash_results.csv","w"); fprintf(csv,"N,d,naive_ms,flash_ms,speedup\n");
    cudaEvent_t evs,eve; cudaEventCreate(&evs); cudaEventCreate(&eve);
    int B=1,H=8,d=HEAD_DIM;
    int seqs[]={128,256,512,1024};
    for(int i=0;i<4;++i){
        printf("\n── N=%d ────────────────────────\n",seqs[i]);
        benchmark(seqs[i],d,B,H,evs,eve,3,20,csv);
    }
    fclose(csv); printf("\nSaved flash_results.csv\n"); return 0;
}
'''

with open('flash_attn_kernels.cu','w') as f: f.write(kernels_code)
print('Written flash_attn_kernels.cu')

In [ ]:
# ── Cell 3: Compile and run standalone benchmark ──────────────────────
!nvcc -O2 -arch=sm_75 --maxrregcount=64 --use_fast_math \
      flash_attn_kernels.cu -lcublas -o flash_bench 2>&1
print('Compiled!')

In [ ]:
# ── Cell 4: Run benchmark ─────────────────────────────────────────────
!./flash_bench

In [ ]:
# ── Cell 5: Correctness check vs PyTorch ─────────────────────────────
import torch
import torch.nn.functional as F

# We test the Python online_softmax algorithm first
def online_softmax_python(x):
    B, N = x.shape
    m = torch.full((B,), float('-inf'), device=x.device)
    l = torch.zeros(B, device=x.device)
    for j in range(N):
        x_j = x[:, j]
        m_new = torch.maximum(m, x_j)
        l = l * torch.exp(m - m_new) + torch.exp(x_j - m_new)
        m = m_new
    return torch.exp(x - m.unsqueeze(1)) / l.unsqueeze(1)

x = torch.randn(4, 256)
ref = F.softmax(x, dim=-1)
ours = online_softmax_python(x)
print(f'Online softmax max error vs F.softmax: {(ref - ours).abs().max():.2e}  (should be < 1e-5)')

# Test pure-PyTorch attention (our reference implementation)
def pytorch_flash_reference(Q, K, V):
    """Pure PyTorch. Same result as F.scaled_dot_product_attention."""
    scale = Q.size(-1) ** -0.5
    scores = torch.matmul(Q, K.transpose(-2, -1)) * scale
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V)

B, H, N, d = 1, 4, 256, 64
Q = torch.randn(B, H, N, d, device='cuda')
K = torch.randn(B, H, N, d, device='cuda')
V = torch.randn(B, H, N, d, device='cuda')

with torch.no_grad():
    ref_pt = F.scaled_dot_product_attention(Q, K, V)
    ref_ours = pytorch_flash_reference(Q, K, V)

print(f'Our reference vs PyTorch: max_err={( ref_pt - ref_ours).abs().max():.2e}  (should be < 1e-4)')

In [ ]:
# ── Cell 6: Build PyTorch extension ──────────────────────────────────
# Write setup.py
setup_code = '''
from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CUDAExtension
import torch

cap = torch.cuda.get_device_capability()
arch = f"sm_{cap[0]}{cap[1]}"
print(f"Building for {arch}")

setup(
    name="flash_attn_cuda",
    ext_modules=[CUDAExtension(
        name="flash_attn_cuda",
        sources=["flash_attn_binding.cpp","flash_attn_kernels.cu","flash_attn_launch.cu"],
        extra_compile_args={
            "cxx":["-O2","-std=c++17"],
            "nvcc":["-O2",f"-arch={arch}","--maxrregcount=64","--use_fast_math","-std=c++17"]
        }
    )],
    cmdclass={"build_ext": BuildExtension}
)
'''
with open('setup.py','w') as f: f.write(setup_code)

# Write binding + launch files (simplified versions)
binding_code = '''
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <cuda_runtime.h>
#include <vector>

namespace py = pybind11;

void launch_flash_attention(
    const float* Q, const float* K, const float* V,
    float* O, float* L,
    int B, int H, int N, int d,
    cudaStream_t stream
);

std::vector<torch::Tensor> flash_fwd(
    torch::Tensor Q,
    torch::Tensor K,
    torch::Tensor V
){
    TORCH_CHECK(Q.is_cuda(), "CUDA required");
    TORCH_CHECK(Q.dtype() == torch::kFloat32, "float32 required");

    Q = Q.contiguous();
    K = K.contiguous();
    V = V.contiguous();

    int B = Q.size(0);
    int H = Q.size(1);
    int N = Q.size(2);
    int d = Q.size(3);

    auto O = torch::zeros_like(Q);
    auto L = torch::zeros({B, H, N}, Q.options());

    auto stream = at::cuda::getCurrentCUDAStream();

    launch_flash_attention(
        Q.data_ptr<float>(),
        K.data_ptr<float>(),
        V.data_ptr<float>(),
        O.data_ptr<float>(),
        L.data_ptr<float>(),
        B, H, N, d,
        stream
    );

    return std::vector<torch::Tensor>{O, L};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &flash_fwd, "Flash Attention forward");
}
'''
with open('flash_attn_binding.cpp','w') as f: f.write(binding_code)

launch_code = '''
#include <cuda_runtime.h>
#define Br 16
#define Bc 16
__global__ void flash_attention_fwd(const float*Q,const float*K,const float*V,
    float*O,float*L,int N,int d,int H);

void launch_flash_attention(const float*Q,const float*K,const float*V,
    float*O,float*L,int B,int H,int N,int d,cudaStream_t stream)
{
    int qt=(N+Br-1)/Br,tbh=B*H;
    dim3 grid(qt,tbh),block(Br*Bc);
    size_t smem=(size_t)(Br*d+Bc*d+Bc*d+Br*Bc)*sizeof(float);
    flash_attention_fwd<<<grid,block,smem,stream>>>(Q,K,V,O,L,N,d,H);
}
'''
with open('flash_attn_launch.cu','w') as f: f.write(launch_code)
print('Source files written')

In [ ]:
# ── Cell 7: Install the PyTorch extension ─────────────────────────────
!pip install -e . --quiet 2>&1 | tail -5
print('Extension installed!')

In [ ]:
# ── Cell 8: Test the PyTorch extension ───────────────────────────────
import importlib, torch
import torch.nn.functional as F

import flash_attn_cuda
print('Extension imported ✓')

B, H, N, d = 1, 4, 256, 64
Q = torch.randn(B, H, N, d, device='cuda', dtype=torch.float32)
K = torch.randn(B, H, N, d, device='cuda', dtype=torch.float32)
V = torch.randn(B, H, N, d, device='cuda', dtype=torch.float32)

# Run our CUDA kernel
O_flash, L = flash_attn_cuda.forward(Q, K, V)

# PyTorch reference
with torch.no_grad():
    O_ref = F.scaled_dot_product_attention(Q, K, V)

max_err = (O_flash - O_ref).abs().max().item()
mean_err = (O_flash - O_ref).abs().mean().item()
print(f'Max error  vs PyTorch: {max_err:.4e}  (atol 1e-2 OK for float32)')
print(f'Mean error vs PyTorch: {mean_err:.4e}')
print(f'allclose (atol=1e-2): {torch.allclose(O_flash, O_ref, atol=1e-2)}')

In [ ]:
# ── Cell 9: Autograd backward test ───────────────────────────────────
import torch
import torch.nn.functional as F

class FlashAttnFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Q, K, V):
        O, L = flash_attn_cuda.forward(Q.float().contiguous(),
                                        K.float().contiguous(),
                                        V.float().contiguous())
        ctx.save_for_backward(Q, K, V, O, L)
        return O

    @staticmethod
    def backward(ctx, dO):
        Q, K, V, O, L = ctx.saved_tensors
        scale = Q.size(-1) ** -0.5
        scores = torch.matmul(Q, K.transpose(-2, -1)) * scale
        P = torch.exp(scores - L.unsqueeze(-1))
        dV = torch.matmul(P.transpose(-2, -1), dO)
        dP = torch.matmul(dO, V.transpose(-2, -1))
        dS = P * (dP - (dP * P).sum(dim=-1, keepdim=True))
        dQ = torch.matmul(dS, K) * scale
        dK = torch.matmul(dS.transpose(-2, -1), Q) * scale
        return dQ, dK, dV

# Test backward
B, H, N, d = 1, 2, 64, 64
Q = torch.randn(B, H, N, d, device='cuda', requires_grad=True)
K = torch.randn(B, H, N, d, device='cuda', requires_grad=True)
V = torch.randn(B, H, N, d, device='cuda', requires_grad=True)

O = FlashAttnFn.apply(Q, K, V)
loss = O.sum()
loss.backward()

print(f'dQ shape: {Q.grad.shape}  norm: {Q.grad.norm():.4f}')
print(f'dK shape: {K.grad.shape}  norm: {K.grad.norm():.4f}')
print(f'dV shape: {V.grad.shape}  norm: {V.grad.norm():.4f}')
print('Backward pass PASSED ✓')

In [ ]:
# ── Cell 10: Python benchmark + charts ───────────────────────────────
import time
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

def benchmark_attention(B=1, H=8, d=64, warmup=3, reps=20):
    results = []
    print(f'\n{"N":>6}  {"PyTorch ref":>14}  {"Flash ours":>12}  {"Speedup":>9}  '
          f'{"PyTorch mem MB":>14}  {"Flash mem MB":>12}')
    print('─' * 75)

    for N in [128, 256, 512, 1024]:
        Q = torch.randn(B,H,N,d,device='cuda',dtype=torch.float32)
        K = torch.randn(B,H,N,d,device='cuda',dtype=torch.float32)
        V = torch.randn(B,H,N,d,device='cuda',dtype=torch.float32)

        for _ in range(warmup):
            with torch.no_grad():
                F.scaled_dot_product_attention(Q,K,V)
                flash_attn_cuda.forward(Q,K,V)
        torch.cuda.synchronize()

        # PyTorch ref
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(reps):
            with torch.no_grad(): r = F.scaled_dot_product_attention(Q,K,V)
        torch.cuda.synchronize()
        pt_ms  = (time.perf_counter()-t0)*1000/reps
        pt_mem = torch.cuda.max_memory_allocated()/1e6

        # Flash
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(reps):
            with torch.no_grad(): flash_attn_cuda.forward(Q,K,V)
        torch.cuda.synchronize()
        fl_ms  = (time.perf_counter()-t0)*1000/reps
        fl_mem = torch.cuda.max_memory_allocated()/1e6

        su = pt_ms/fl_ms
        results.append((N,pt_ms,fl_ms,su,pt_mem,fl_mem))
        mark = ' ← faster' if su > 1.05 else ''
        print(f'{N:>6}  {pt_ms:>14.3f}  {fl_ms:>12.3f}  {su:>9.2f}x  {pt_mem:>14.1f}  {fl_mem:>12.1f}{mark}')

    return results

results = benchmark_attention()

# ── Plot ──────────────────────────────────────────────────────────────
Ns     = [r[0] for r in results]
pt_ms  = [r[1] for r in results]
fl_ms  = [r[2] for r in results]
speedup= [r[3] for r in results]
pt_mem = [r[4] for r in results]
fl_mem = [r[5] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Latency
x=np.arange(len(Ns)); w=0.35
axes[0].bar(x-w/2,pt_ms,w,label='PyTorch ref',color='#888',alpha=0.85)
axes[0].bar(x+w/2,fl_ms,w,label='Flash (ours)',color='#2a7abf',alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(Ns)
axes[0].set_yscale('log'); axes[0].legend(); axes[0].grid(axis='y',alpha=0.3)
axes[0].set_title('Latency (ms, log scale)',fontweight='bold')
axes[0].set_xlabel('Sequence length N')

# Speedup
bc=['#1a9e5c' if s>1 else '#e07b39' for s in speedup]
bars=axes[1].bar(x,speedup,0.5,color=bc,alpha=0.85)
axes[1].axhline(1,color='#888',linestyle='--',linewidth=1.2)
for bar,s in zip(bars,speedup):
    axes[1].text(bar.get_x()+bar.get_width()/2,s+0.02,f'{s:.2f}x',
                 ha='center',fontsize=9,fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(Ns)
axes[1].set_title('Speedup vs PyTorch ref',fontweight='bold')
axes[1].set_xlabel('Sequence length N'); axes[1].grid(axis='y',alpha=0.3)

# Memory
axes[2].plot(Ns,pt_mem,'o-',color='#888',linewidth=2,markersize=7,label='PyTorch (O(N²) mat)')
axes[2].plot(Ns,fl_mem,'s-',color='#2a7abf',linewidth=2,markersize=7,label='Flash (O(Nd) SRAM)')
axes[2].set_xlabel('Sequence length N'); axes[2].set_ylabel('Peak GPU mem (MB)')
axes[2].set_title('Memory: O(N²) vs O(Nd)',fontweight='bold')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('flash_benchmark.png',dpi=150,bbox_inches='tight')
plt.show()
print('Saved flash_benchmark.png — paste this in your README!')

In [ ]:
# ── Cell 11: Memory growth visualisation ─────────────────────────────
# Shows the key insight: naive grows as N², flash stays flat
import numpy as np, matplotlib.pyplot as plt

Ns = np.array([64,128,256,512,1024,2048,4096])
d, B, H = 64, 1, 8

# Naive: N² × B × H × sizeof(float) MB
naive_mem = B * H * Ns**2 * 4 / 1e6

# Flash: only tiles in SRAM — effectively constant per SM
# 2 tiles × (Br+Bc) × d × sizeof(float) per SM call
Br, Bc = 16, 16
flash_mem = np.ones_like(Ns) * (Br*d + Bc*d + Bc*d + Br*Bc) * 4 / 1024  # KB

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(Ns, naive_mem, 'o-', color='#e07b39', linewidth=2, markersize=7, label='Naive O(N²) HBM')
ax1.axhline(1000, color='#2a7abf', linestyle='--', linewidth=1.5, label='Flash (O(Nd) — effectively flat)')
ax1.set_xlabel('Sequence length N'); ax1.set_ylabel('Memory for attention scores (MB)')
ax1.set_title('Why memory matters: O(N²) explodes', fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3)
for n, m in zip(Ns, naive_mem):
    if m > 1: ax1.annotate(f'{m:.0f}MB', (n, m), textcoords='offset points', xytext=(5,5), fontsize=8)

# Annotation table
ax2.axis('off')
table_data = []
for n, nm in zip(Ns, naive_mem):
    sm = (Br*d + Bc*d + Bc*d + Br*Bc) * 4 / 1024
    table_data.append([f'{n}', f'{nm:.1f} MB', f'{sm:.1f} KB', 'SRAM only ✓' if nm > 10 else '~same'])

table = ax2.table(cellText=table_data,
                   colLabels=['N', 'Naive HBM', 'Flash SRAM', 'Flash wins'],
                   loc='center', cellLoc='center')
table.auto_set_font_size(True)
table.scale(1, 1.5)
ax2.set_title('Memory comparison table', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved memory_comparison.png')

In [ ]:
# ── Cell 12: Auto-generate README table ──────────────────────────────
import pandas as pd
df = pd.read_csv('flash_results.csv')
print('Copy this into your README:\n')
print('| N | d | Naive (ms) | Flash (ms) | Speedup | Memory naive | Memory flash |')
print('|---|---|------------|------------|---------|--------------|--------------|')
for _, row in df.iterrows():
    n = int(row.N); d = int(row.d)
    naive_mem = 8 * n * n * 4 / 1e6  # B=1, H=8
    print(f'| {n} | {d} | {row.naive_ms:.3f} | {row.flash_ms:.3f} | **{row.speedup:.2f}x** | {naive_mem:.1f} MB | ~34 KB (SRAM) |')